# MScFE 600 Financial Data — Group Work Project 1
## IVV technical-indicator analysis and alternative-data exploration

This notebook supports the written assignment by (1) documenting the paper's data and methodology, (2) analyzing the iShares Core S&P 500 ETF (IVV), (3) applying a Pearson-correlation feature-selection approach with chronological validation, and (4) providing an exploratory guide to Search Volume Index (SVI) data for groups completing Part 2.

**Replication choice.** The reference paper analyzes daily ETF data from 12 December 2009 to 1 January 2020. The download cell uses that period so results are comparable in scope. Yahoo Finance treats the end date as exclusive, so the code requests data through 2 January 2020.

**Important distinction.** The paper calculates 216 features. This notebook uses a transparent, compact set of eight technical indicators and Pearson correlation, one of the simpler metrics allowed by the assignment. It is an instructional partial replication, not an exact reproduction of the 216-feature experiment. Results are calculated when the notebook is run; no model metrics are hard-coded.

Complete the work collaboratively, confirm that the results make sense, and follow WQU's academic-integrity and AI-use policies. If your group has one or two members, Part 2 is not required by the assignment.

## Run instructions and reproducibility

Run cells from top to bottom in Google Colab or Jupyter. The notebook downloads prices from Yahoo Finance and installs the Python packages it needs. Internet access is therefore required for the download. If the provider changes its data format or is unavailable, record the issue rather than substituting invented results.

The validation design uses five expanding, chronological folds. Feature selection is recalculated inside each training fold, and standardization is fitted only on training data. Features for date *t* are lagged one trading day to predict the paper's directional label for date *t*, reducing look-ahead leakage. The paper reports ten-fold cross-validation; the walk-forward setup here is a deliberate finance-oriented adaptation and must be described as such in the report.

In [ ]:
%pip install -q yfinance scikit-learn seaborn

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

RANDOM_STATE = 42
TICKER = "IVV"
START_DATE = "2009-12-12"
END_DATE = "2020-01-02"  # exclusive; includes the final trading session of 2019
N_SPLITS = 5
TOP_N_FEATURES = 5

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print(f"Configuration: {TICKER}, {START_DATE} to {END_DATE} (end exclusive)")

In [ ]:
# Download the six base fields used by the paper: OHLC, adjusted close, and volume.
raw = yf.download(
    TICKER,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    actions=False,
    progress=False,
    threads=False
)

if raw.empty:
    raise RuntimeError(
        "Yahoo Finance returned no data. Check the internet connection, ticker, and date range."
    )

# yfinance may return MultiIndex columns, even for one ticker. Flatten safely.
if isinstance(raw.columns, pd.MultiIndex):
    known_fields = {"Open", "High", "Low", "Close", "Adj Close", "Volume"}
    flat_columns = []
    for column in raw.columns:
        parts = [str(part) for part in column]
        field = next((part for part in parts if part in known_fields), parts[0])
        flat_columns.append(field)
    raw.columns = flat_columns

raw = raw.loc[:, ~raw.columns.duplicated()].copy()
required_fields = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
missing_fields = [field for field in required_fields if field not in raw.columns]
if missing_fields:
    raise ValueError(f"Required Yahoo Finance columns are missing: {missing_fields}")

data = raw[required_fields].sort_index().copy()
data.index = pd.to_datetime(data.index)
data.index.name = "Date"
data = data.dropna(subset=required_fields)

print(f"Rows downloaded: {len(data):,}")
print(f"First observation: {data.index.min().date()}")
print(f"Last observation:  {data.index.max().date()}")
display(data.head())

## Part 1 — Q1 and Q2: Data and security understanding

### Q1. Data understanding

Sagaceta-Mejía, Sánchez-Gutiérrez, and Fresán-Figueroa obtain daily market observations from Yahoo Finance for three ETFs: ECH, EWZ, and IVV. The six base fields are Open, High, Low, Close, Adjusted Close, and Volume. They use Pandas Technical Analysis (Pandas TA) to calculate another 210 technical indicators, creating 216 features in total (sections 2.1–2.3).

Technical indicators transform raw price and volume history into signals about trend, momentum, volatility, and trading activity. Examples include moving averages, Relative Strength Index (RSI), and band-based volatility measures. A moving average smooths short-term noise; RSI compares recent upward and downward price changes; volatility bands describe a price's distance from its recent average. Indicators can summarize different patterns and help a model discriminate among market states, but they are not guarantees of future returns. Many indicators are redundant, and selecting features on the full dataset can cause information leakage.

### Q2. Security understanding — IVV

IVV is the iShares Core S&P 500 ETF, an exchange-traded fund designed to track large-capitalization U.S. equities represented by the S&P 500 Index. Unlike buying one company, an S&P 500 ETF provides diversified exposure to a broad basket of large U.S. companies. Its price is affected by aggregate earnings expectations, interest rates, economic conditions, index composition, and the market prices of its holdings.

The next cell calculates the fund's price history and basic statistics for the paper's 2009–2020 analysis window. These are **sample-period statistics**, not current fund assets under management or today's market price. In the report, use the generated table and labeled plot, then keep the prose description to the assignment's strict one-page limit.

**Why classification rather than regression?** The paper studies market direction, not the exact future price. Its binary label is Gamma(t) = 1 when Open(t) > Open(t−1), and 0 otherwise (section 2.4). Directional labels can be more directly aligned with a buy/sell decision than predicting an exact price level, which contains scale and magnitude noise.

Two alternative labels are:
1. **Close-to-close direction:** 1 if AdjustedClose(t) > AdjustedClose(t−1), otherwise 0.
2. **Three-class return:** +1 if the next-day return exceeds a chosen positive threshold, −1 if it is below the negative threshold, and 0 otherwise. The threshold should be specified before evaluation.

Classification is not always preferable: regression can be useful when forecasts of return magnitude, rather than direction alone, are required.

In [ ]:
# Descriptive price statistics for the selected sample period.
adj = data["Adj Close"].dropna()
daily_returns = adj.pct_change().dropna()
years = max((adj.index[-1] - adj.index[0]).days / 365.25, 1 / 365.25)
wealth = adj / adj.iloc[0]
drawdown = wealth / wealth.cummax() - 1

fund_stats = pd.DataFrame({
    "Measure": [
        "Observations", "Start date", "End date",
        "First adjusted close (USD)", "Last adjusted close (USD)",
        "Total adjusted-price return", "Annualized return (CAGR)",
        "Annualized volatility", "Maximum drawdown",
        "Mean daily return", "Median daily return"
    ],
    "Value": [
        len(data),
        data.index.min().date().isoformat(),
        data.index.max().date().isoformat(),
        round(float(adj.iloc[0]), 2),
        round(float(adj.iloc[-1]), 2),
        f"{(adj.iloc[-1] / adj.iloc[0] - 1):.2%}",
        f"{((adj.iloc[-1] / adj.iloc[0]) ** (1 / years) - 1):.2%}",
        f"{(daily_returns.std(ddof=1) * np.sqrt(252)):.2%}",
        f"{drawdown.min():.2%}",
        f"{daily_returns.mean():.4%}",
        f"{daily_returns.median():.4%}"
    ]
})
display(fund_stats)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(adj.index, adj, linewidth=1.6, label="IVV adjusted close")
ax.set(title="IVV adjusted-price history (paper sample window)", xlabel="Date", ylabel="Adjusted close (USD)")
ax.legend()
fig.tight_layout()
plt.show()

print("Open-price descriptive statistics:")
display(data["Open"].describe().rename("Open (USD)").to_frame())

## Q3. Methodology understanding — a clearer section structure

A useful reorganization of the paper separates what the dataset contains from how the authors analyze it.

### Proposed Section 2 — Data
- **2.1 Stocks analyzed and sample period:** ECH, EWZ, and IVV; 12 December 2009 to 1 January 2020.
- **2.2 Data source and raw variables:** daily OHLC, Adjusted Close, and Volume from Yahoo Finance.
- **2.3 Technical-indicator construction:** generation of the 210 derived indicators from the six base fields.
- **2.4 Class assignment:** the binary Gamma label based on consecutive opening prices.
- **2.5 Data normalization:** min–max scaling of feature values to a common range.
- **2.6 Data cleaning:** removal of rows with missing values created by indicators or unavailable market data.
- **2.7 Descriptive statistics and class distribution:** summarize data ranges, distributions, and label balance.

### Proposed Section 3 — Methodology
- **3.1 CRISP-DM workflow:** business understanding, data understanding, preparation, modeling, evaluation, and presentation.
- **3.2 Statistical feature-selection measures:** Pearson correlation, distributional/statistical rankings such as variance, skewness, kurtosis and mean absolute difference, and dispersion ratio.
- **3.3 Model-based feature selection:** LASSO regression, tree-based selection, and Principal Feature Analysis.
- **3.4 Feature-set construction and overlap:** form Selected(n) sets from features that recur across ranking subsets; use Jaccard distance to compare sets.
- **3.5 Neural-network model:** the multilayer perceptron (MLP) used to classify the directional label.
- **3.6 Validation and experiment design:** ten-fold cross-validation in the paper, feature-count experiments, average accuracy, and training time.

Descriptive statistics belong in Data/Exploratory Analysis. Pearson correlation can be presented as descriptive analysis, but when it is used to rank/select predictors it also becomes part of the feature-selection method. LASSO is a model-based selection method because its fitted coefficients are penalized; it is not merely a descriptive statistic.

**Feature optimization in the paper.** The authors evaluate several feature-selection techniques, rank or retain salient indicators, construct feature subsets based on repeated selection, and evaluate those subsets with an MLP. Their abstract reports that using about 5% of the technical indicators substantially reduces model-training cost and improves accuracy. For IVV, their Table 6 reports 71.10% accuracy with all 216 features and 78.46% with the selected nine features. Those are the paper's reported results, not results from this compact notebook.

## Q4. Feature understanding

A **feature** is an input variable supplied to a model. In this notebook, examples include RSI, MACD, the ratio of close to a 20-day moving average, Bollinger-band width, rolling volatility, and 10-day rate of change.

A **method** is a procedure used to prepare, select, or analyze features. Examples include Pearson-correlation ranking, LASSO, min–max normalization, and cross-validation. A **model** is the fitted mapping from inputs to an outcome; the paper's model is a multilayer perceptron (MLP) neural network.

Useful feature categories include:
- **Raw features:** Open, High, Low, Close, Adjusted Close, and Volume.
- **Engineered/technical features:** indicators computed from raw market data.
- **Lagged features:** earlier observations used to forecast a later target.
- **External/alternative features:** information such as search volume or news sentiment.
- **Target/label:** the outcome to be predicted; it is not an input feature.

The paper's indicator optimization aims to reduce redundancy and computational cost while preserving or improving predictive accuracy. This notebook implements only one of the simpler permitted approaches—Pearson correlation—and recomputes the ranking within each training fold.

## Q5. Optimization understanding

**Cross-validation** evaluates a modeling process on multiple train/validation partitions rather than relying on one split. Results across partitions help show whether performance is sensitive to a particular sample.

**K-fold cross-validation** divides observations into *k* folds. A model is fitted *k* times; each fold is used for validation once while the other folds are used for training. Standard shuffled K-fold can leak future information when used carelessly on financial time series. This notebook uses expanding-window TimeSeriesSplit folds: each test block occurs strictly after its training block. This is a deliberate adaptation of the paper's ten-fold procedure.

**Jaccard distance** measures the dissimilarity of two sets. For feature sets A and B, dJ(A,B) = 1 − |A ∩ B| / |A ∪ B|. The value is 0 for identical sets and 1 when they have no features in common. In this notebook it measures the stability of the top-five feature set between consecutive validation folds.

Compared with Jaccard distance:
- **Euclidean distance** is the square root of the sum of squared coordinate differences and is suitable for numeric vectors where magnitude matters.
- **Manhattan distance** is the sum of absolute coordinate differences and is also defined on numeric vectors.
Jaccard distance is especially interpretable for membership/overlap sets; Euclidean and Manhattan distance compare numeric coordinates and can be scale-sensitive.

**What is an optimal solution?** In the paper's experimental setup, candidate feature subsets are compared using average cross-validated MLP accuracy and computational cost. A practical optimum is a small, sufficiently predictive feature set—not simply the subset with the greatest number of features. The choice balances accuracy, training time, and feature stability; it is not a guarantee of future investment returns.

### Step 1 — Financial problem
The paper aims to select informative technical indicators to classify ETF market direction more efficiently. Emerging markets may have different liquidity, volatility, information flow, trading costs, and market structure from developed markets. These differences can affect how reliably indicators generalize and motivate testing features on the specific market instruments rather than assuming one feature set works everywhere.

### Step 2 — Main application takeaways
The paper reports that feature selection can reduce computation and improve directional classification accuracy. In its IVV result, accuracy increases from 71.10% using 216 features to 78.46% using nine selected features. The authors identify several technical indicators as useful, including Even Better SineWave, Balance of Power, Stochastic RSI, a correlation-trend indicator, and KDJ (see Appendix A). These findings describe the study's sample, not a live trading guarantee.

In [ ]:
# Build a compact set of interpretable technical features from the downloaded OHLCV data.
features_data = data.copy()
close = features_data["Close"]
high = features_data["High"]
low = features_data["Low"]
volume = features_data["Volume"]

delta = close.diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
rs = avg_gain / avg_loss.replace(0, np.nan)
features_data["RSI_14"] = 100 - (100 / (1 + rs))
features_data.loc[(avg_loss == 0) & (avg_gain > 0), "RSI_14"] = 100

ema12 = close.ewm(span=12, min_periods=12, adjust=False).mean()
ema26 = close.ewm(span=26, min_periods=26, adjust=False).mean()
features_data["MACD"] = ema12 - ema26
features_data["MACD_Signal"] = features_data["MACD"].ewm(
    span=9, min_periods=9, adjust=False
).mean()
features_data["MACD_Hist"] = features_data["MACD"] - features_data["MACD_Signal"]

ma20 = close.rolling(20, min_periods=20).mean()
std20 = close.rolling(20, min_periods=20).std(ddof=0)
upper = ma20 + 2 * std20
lower = ma20 - 2 * std20
features_data["Close_MA20_Ratio"] = close / ma20 - 1
features_data["BB_Width_20"] = (upper - lower) / ma20

features_data["Daily_Log_Return"] = np.log(
    features_data["Adj Close"] / features_data["Adj Close"].shift(1)
)
features_data["Rolling_Volatility_20"] = (
    features_data["Daily_Log_Return"].rolling(20, min_periods=20).std() * np.sqrt(252)
)
features_data["ROC_10"] = close.pct_change(10)
features_data["Volume_Change"] = volume.pct_change().replace([np.inf, -np.inf], np.nan)

feature_cols = [
    "RSI_14", "MACD", "MACD_Signal", "MACD_Hist",
    "Close_MA20_Ratio", "BB_Width_20", "Rolling_Volatility_20", "ROC_10"
]

# Paper's class definition: 1 when today's Open exceeds yesterday's Open, otherwise 0.
features_data["Gamma"] = (features_data["Open"] > features_data["Open"].shift(1)).astype("int8")

# Use indicators from t-1 to predict Gamma at t. This preserves chronological availability.
model_data = features_data[feature_cols].shift(1)
model_data["Gamma"] = features_data["Gamma"]
model_data = model_data.replace([np.inf, -np.inf], np.nan).dropna()

X = model_data[feature_cols].astype(float)
y = model_data["Gamma"].astype(int)

if len(model_data) < 100:
    raise ValueError("Too few complete observations after feature engineering for reliable validation.")

print(f"Usable modeling rows: {len(model_data):,}")
print(f"Input features: {len(feature_cols)}")
print("Target balance (0 = opening price did not rise; 1 = opening price rose):")
display(y.value_counts(normalize=True).rename("Share").to_frame().round(3))
display(model_data[feature_cols + ["Gamma"]].head())

In [ ]:
# Paper-inspired descriptive figures: adjusted price and cumulative positive-direction labels.
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(data.index, data["Adj Close"], linewidth=1.5)
axes[0].set_title("IVV adjusted-close history")
axes[0].set_ylabel("Adjusted close (USD)")

gamma = features_data["Gamma"].loc[model_data.index]
axes[1].plot(gamma.index, gamma.cumsum(), linewidth=1.5)
axes[1].set_title("Cumulative count of positive opening-price movements")
axes[1].set_ylabel("Cumulative Gamma = 1")
axes[1].set_xlabel("Date")

fig.tight_layout()
plt.show()

## Step 3 — Replication: Pearson correlation + chronological validation

The assignment permits choosing an easier metric such as correlation or dispersion ratio rather than reproducing every selection method in the paper. Here, absolute Pearson correlation with the binary target ranks candidate technical indicators. Because the target is binary, the Pearson coefficient is also interpretable as a point-biserial association.

For every fold:
1. Fit a majority-class baseline on training data.
2. Rank features using only the training partition.
3. Fit an MLP using all eight features and another MLP using the five highest-ranked features. Fit each scaler on training data only.
4. Evaluate only on the future test partition.
5. Compare selected-feature sets across successive folds using Jaccard distance.

The results will differ from the paper because this exercise uses eight indicators rather than 216 and five chronological test folds rather than the paper's ten-fold setup. Do not describe it as an exact replication.

In [ ]:
# Out-of-sample evaluation. Rankings and scaling are recomputed within each training fold.
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
fold_results = []
feature_rank_records = []
oos_prediction_frames = []
previous_selected = None

def make_mlp():
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="adam",
            alpha=0.001,
            max_iter=500,
            random_state=RANDOM_STATE
        )
    )

for fold_number, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Pearson correlation ranking is learned from training observations only.
    train_corr = X_train.join(y_train.rename("Gamma")).corr(method="pearson")["Gamma"]
    ranking = train_corr.drop("Gamma").abs().sort_values(ascending=False)
    selected_features = ranking.head(TOP_N_FEATURES).index.tolist()

    for feature_name, score in ranking.items():
        feature_rank_records.append({
            "Fold": fold_number,
            "Feature": feature_name,
            "Absolute Pearson correlation": float(score)
        })

    baseline = DummyClassifier(strategy="most_frequent")
    baseline.fit(X_train, y_train)
    baseline_pred = baseline.predict(X_test)

    full_model = make_mlp()
    selected_model = make_mlp()

    # Warnings are retained by default so convergence issues remain visible.
    full_model.fit(X_train, y_train)
    selected_model.fit(X_train[selected_features], y_train)

    full_pred = full_model.predict(X_test)
    selected_pred = selected_model.predict(X_test)

    current_set = set(selected_features)
    if previous_selected is None:
        feature_jaccard_distance = np.nan
    else:
        union = current_set | previous_selected
        intersection = current_set & previous_selected
        feature_jaccard_distance = 1 - len(intersection) / len(union) if union else 0.0

    fold_results.append({
        "Fold": fold_number,
        "Test start": X_test.index.min().date().isoformat(),
        "Test end": X_test.index.max().date().isoformat(),
        "Baseline accuracy": accuracy_score(y_test, baseline_pred),
        "All 8 features accuracy": accuracy_score(y_test, full_pred),
        "Top 5 accuracy": accuracy_score(y_test, selected_pred),
        "Top 5 balanced accuracy": balanced_accuracy_score(y_test, selected_pred),
        "Top 5 precision": precision_score(y_test, selected_pred, zero_division=0),
        "Top 5 recall": recall_score(y_test, selected_pred, zero_division=0),
        "Top 5 F1": f1_score(y_test, selected_pred, zero_division=0),
        "Feature-set Jaccard distance": feature_jaccard_distance,
        "Selected features": ", ".join(selected_features)
    })

    oos_prediction_frames.append(pd.DataFrame({
        "Actual Gamma": y_test,
        "Predicted Gamma": pd.Series(selected_pred, index=y_test.index),
        "Fold": fold_number
    }, index=y_test.index))

    previous_selected = current_set
    print(f"Finished fold {fold_number}/{N_SPLITS}: "
          f"{X_test.index.min().date()} to {X_test.index.max().date()}")

results_df = pd.DataFrame(fold_results)
oos_predictions = pd.concat(oos_prediction_frames).sort_index()
feature_rankings = pd.DataFrame(feature_rank_records)

print("Out-of-sample performance by chronological fold")
display(results_df.round(4))

metric_cols = [
    "Baseline accuracy", "All 8 features accuracy", "Top 5 accuracy",
    "Top 5 balanced accuracy", "Top 5 precision", "Top 5 recall", "Top 5 F1"
]
summary_df = results_df[metric_cols].mean().rename("Mean across folds").to_frame()
display(summary_df.round(4))

In [ ]:
# Plot fold-level performance and feature-selection stability.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

performance_long = results_df.melt(
    id_vars="Fold",
    value_vars=["Baseline accuracy", "All 8 features accuracy", "Top 5 accuracy"],
    var_name="Approach",
    value_name="Accuracy"
)
sns.lineplot(data=performance_long, x="Fold", y="Accuracy", hue="Approach", marker="o", ax=axes[0])
axes[0].set(title="Out-of-sample accuracy by fold", xlabel="Chronological fold", ylabel="Accuracy")
axes[0].set_ylim(0, 1)

mean_abs_corr = (
    feature_rankings.groupby("Feature")["Absolute Pearson correlation"]
    .mean()
    .sort_values()
)
mean_abs_corr.plot(kind="barh", ax=axes[1])
axes[1].set(title="Mean absolute Pearson correlation across training folds",
            xlabel="Mean absolute correlation", ylabel="Feature")

fig.tight_layout()
plt.show()

# Correlation between the lagged input indicators (exploratory, not the selection rule).
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(X.corr(), annot=True, fmt=".2f", center=0, cmap="coolwarm",
            vmin=-1, vmax=1, square=True, ax=ax,
            cbar_kws={"label": "Pearson correlation"})
ax.set_title("Correlation among lagged technical features")
fig.tight_layout()
plt.show()

# Overall out-of-sample confusion matrix for the top-five feature model.
cm = confusion_matrix(oos_predictions["Actual Gamma"], oos_predictions["Predicted Gamma"], labels=[0, 1])
display(pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Gamma = 0", "Gamma = 1"]).plot()
plt.title("Out-of-sample confusion matrix — top-five correlation features")
plt.tight_layout()
plt.show()

## Interpreting your empirical results

Use the generated tables and charts—not the paper's published figures—to write the replication result section.

- Compare the chronological majority-class baseline, all-eight-feature MLP, and top-five-feature MLP. A higher score in one fold does not establish consistent improvement.
- Report mean fold accuracy and also balanced accuracy, precision, recall, and F1 so that class imbalance is visible.
- Read the feature-set Jaccard distance as a **stability** measure: a lower value means more overlap with the previous fold's selected set.
- Treat convergence warnings as a diagnostic. If they occur frequently, test a larger iteration limit or simpler network and report the final specification.
- This validation is predictive, not a complete backtest. It excludes transaction costs, slippage, tax, market impact, and portfolio constraints. Do not recommend trading on the basis of these metrics alone.

For a source-value cross-check, the study's Table 6 reports the following IVV figures:

| Published paper experiment | Number of features | Accuracy | Training time | Epochs |
|---|---:|---:|---:|---:|
| Full IVV feature set | 216 | 71.10% | 53.2 seconds | 457 |
| Selected IVV feature set | 9 | 78.46% | 8.68 seconds | 713 |

These numbers are transcribed from Sagaceta-Mejía et al., Table 6; they are not generated by this notebook. The data window, feature count, and validation design differ from the compact Pearson-correlation exercise here.

# Part 2 — Alternative-data user guide: Search Volume Index (SVI)

*Complete this part only if your group has three members, as specified by the assignment.*

## 1. What SVI measures and why it may matter

Sun et al. classify Searching Volume Index (SVI) as behavioral alternative data. Search frequency can provide an observable proxy for public attention to a company, asset, product, or macroeconomic concern. It complements traditional price, volume, and accounting data; it does not replace them. Da, Engelberg, and Gao show that search frequency can capture investor attention and examine its relationship with subsequent stock-price behavior. Attention can be associated with both short-term price pressure and later reversal, so a higher search index should not automatically be interpreted as a positive sentiment signal.

## 2. Data sources and types

Potential sources include Google Trends (Google search topics/terms and geographic/time filters) and Baidu Index for relevant Chinese-language searches. Search data may be available by day or week and by geography or related-query category. The most common output is an index normalized to a range such as 0–100, rather than a count of all searches. Commercial providers may offer higher-frequency or broader coverage, usually for a fee.

For reproducible work, record the exact search term or topic, provider, geography, category, frequency, requested dates, download date, and export settings. Keep the original file and a cleaned version. If several terms are compared, obtain them in a consistent query and geographic context where possible.

## 3. Data quality and preparation

SVI is relative and sample-dependent: 100 means the peak in the selected query window, not 100 searches. Repeated downloads may vary, low-volume terms can be missing or zero, and one word may have unrelated meanings. Weekly and daily frequencies cannot be aligned without an explicit aggregation or resampling rule. Search interest can also be driven by news shocks, holidays, seasonal events, or non-financial events.

Recommended checks include:
- inspect missing dates, missing values, duplicate dates, and out-of-range values;
- confirm whether 0 means no observed interest or insufficient data;
- document the time zone, geography, query, and frequency;
- compare a search series with a relevant financial series only after aligning dates and defining the return horizon;
- avoid look-ahead bias by using only search observations available before the forecast period.

## 4. Ethical, legal, and practical issues

Prefer aggregated data; do not attempt to identify individuals or infer sensitive personal attributes from search behavior. Follow each provider's terms, licensing, attribution requirements, rate limits, and conditions for redistribution. Avoid collecting personal search histories or combining datasets in ways that could re-identify individuals. Retain source provenance, download dates, transformations, and limitations so another analyst can audit the workflow. A publicly viewable index is not automatically unrestricted for all commercial reuse.

## 5. Python ingestion and exploratory analysis

The next cell accepts an exported CSV with a date column and a search-index column. If the file is not present, it creates an explicitly **simulated** sample so the EDA cell can still run end to end. Simulated values demonstrate the workflow only and must never be reported as observed Google Trends data. Replace the sample with a permitted, documented export before making any empirical claim.

The exploratory plots examine the series over time, its distribution, missingness, and—only where an asset-return column is available—the contemporaneous correlation. Correlation alone does not establish that searches cause or reliably predict market returns.

In [ ]:
# SVI user-guide example.
# To use real data, place a permitted CSV named google_trends_svi.csv in the notebook folder.
# Required columns: date and search_index. Optional column: asset_return (e.g., aligned weekly IVV return).
svi_path = Path("google_trends_svi.csv")

if svi_path.exists():
    svi = pd.read_csv(svi_path)
    normalized_columns = {column.lower().strip().replace(" ", "_"): column for column in svi.columns}
    date_column = normalized_columns.get("date") or normalized_columns.get("week") or normalized_columns.get("month")
    index_column = normalized_columns.get("search_index") or normalized_columns.get("search_volume_index") or normalized_columns.get("sv i")
    if date_column is None or index_column is None:
        raise ValueError(
            "CSV needs a date/week/month column and a search_index or search_volume_index column."
        )
    svi = svi.rename(columns={date_column: "date", index_column: "search_index"})
    if "asset_return" not in svi.columns:
        return_column = normalized_columns.get("asset_return") or normalized_columns.get("ivv_return")
        if return_column:
            svi = svi.rename(columns={return_column: "asset_return"})
    svi["date"] = pd.to_datetime(svi["date"], errors="coerce")
    svi["search_index"] = pd.to_numeric(svi["search_index"], errors="coerce")
    if "asset_return" in svi.columns:
        svi["asset_return"] = pd.to_numeric(svi["asset_return"], errors="coerce")
    svi = svi.dropna(subset=["date", "search_index"]).sort_values("date").drop_duplicates("date")
    svi_source_note = "User-supplied CSV (verify source and license before analysis)"
else:
    # Synthetic example only; the values below are not real search or market data.
    rng = np.random.default_rng(RANDOM_STATE)
    demo_dates = pd.date_range("2023-01-01", periods=52, freq="W")
    simulated_index = np.clip(
        48 + 13 * np.sin(np.linspace(0, 4 * np.pi, len(demo_dates))) + rng.normal(0, 7, len(demo_dates)),
        0, 100
    )
    simulated_returns = rng.normal(0.001, 0.02, len(demo_dates))
    svi = pd.DataFrame({
        "date": demo_dates,
        "search_index": simulated_index,
        "asset_return": simulated_returns
    })
    svi_source_note = "SIMULATED demonstration data — not observed Google Trends data"

print(f"SVI source: {svi_source_note}")
display(svi.head())
print("Descriptive statistics:")
display(svi[["search_index"] + (["asset_return"] if "asset_return" in svi.columns else [])].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(svi["date"], svi["search_index"], marker="o", markersize=3, linewidth=1.3)
axes[0].set(title="Search Volume Index over time", xlabel="Date", ylabel="Relative index (typically 0–100)")
sns.histplot(svi["search_index"], bins=12, kde=True, ax=axes[1])
axes[1].set(title="SVI distribution", xlabel="Search index", ylabel="Observations")
fig.tight_layout()
plt.show()

missing_summary = svi.isna().sum().rename("Missing values").to_frame()
display(missing_summary)

if "asset_return" in svi.columns:
    aligned = svi[["search_index", "asset_return"]].dropna()
    if len(aligned) >= 3:
        print(f"Contemporaneous Pearson correlation (descriptive only): "
              f"{aligned['search_index'].corr(aligned['asset_return']):.3f}")
    else:
        print("Not enough aligned observations to compute a meaningful correlation.")

## Works Cited (MLA style)

Da, Zhi, Joseph Engelberg, and Pengjie Gao. “In Search of Attention.” *The Journal of Finance*, vol. 66, no. 5, 2011, pp. 1461–1499. https://doi.org/10.1111/j.1540-6261.2011.01679.x.

Google. “Google Trends.” *Google Trends*, https://trends.google.com/. Accessed 15 Oct. 2026. Cite and document the specific export used in the final report.

Preis, Tobias, Helen Susannah Moat, and H. Eugene Stanley. “Quantifying Trading Behavior in Financial Markets Using Google Trends.” *Scientific Reports*, vol. 3, 2013, article 1684. https://doi.org/10.1038/srep01684.

Sagaceta-Mejía, Alma Rocío, Máximo Eduardo Sánchez-Gutiérrez, and Julián Alberto Fresán-Figueroa. “An Intelligent Approach for Predicting Stock Market Movements in Emerging Markets Using Optimized Technical Indicators and Neural Networks.” *Economics*, vol. 18, no. 1, 2024, article 20220073. https://doi.org/10.1515/econ-2022-0073.

Sun, Yunchuan, et al. “Alternative Data in Finance and Business: Emerging Applications and Theory Analysis (Review).” *Financial Innovation*, vol. 10, 2024, article 127. https://doi.org/10.1186/s40854-024-00652-0.

Yahoo Finance. “iShares Core S&P 500 ETF (IVV) Historical Data.” *Yahoo Finance*, https://finance.yahoo.com/quote/IVV/history/. Accessed 15 Oct. 2026.

**Citation reminder:** cite sources in the written report at the point where their evidence or ideas are used. The Works Cited list alone is not sufficient. The notebook's generated values should be cited as analysis of the stated download, not as values reported by the papers.